In [0]:
%sql
CREATE TABLE sp_catalog.ecomm.sales_analytics
USING CSV
OPTIONS (header = 'true', inferSchema = 'true')
LOCATION 's3://sp-db26-eu-west2-test/E-Commerce Sales Analytics.csv';

In [0]:
# Read from Unity Catalog table
df = spark.table("sp_catalog.ecomm.sales_analytics")

# # Write as Delta to S3 (path-based, unmanaged)
# df.write.format("delta").mode("overwrite") \
#     .save("s3://sp-db26-eu-west2-test/delta_silver/sales_analytics/")



In [0]:
%sql
CREATE SCHEMA sp_catalog.ecomm_silver;

In [0]:
# OR: write + register as a new managed/external UC Delta table in one step
df.write.format("delta").mode("overwrite") \
    .option("path", "s3://sp-db26-eu-west2-test/ecomm_delta_silver/sales_analytics/") \
    .saveAsTable("sp_catalog.ecomm_silver.sales_analytics_delta")

In [0]:
%sql
CREATE SCHEMA sp_catalog.ecomm_silver_iceberg;

In [0]:
df.write.format("iceberg").mode("overwrite") \
    .option("path", "s3://sp-db26-eu-west2-test/ecomm_silver_iceberg/sales_analytics/") \
    .saveAsTable("sp_catalog.ecomm_silver_iceberg.sales_analytics_iceberg")


In [0]:
df.write.format("iceberg").mode("overwrite") \
    .saveAsTable("sp_catalog.ecomm_silver_iceberg.sales_analytics_iceberg")

In [0]:
%sql
-- Interop with Delta (UniForm):
-- Delta Lake tables can be configured with Iceberg reads (Universal Format/UniForm) so Iceberg clients can read Delta data without rewriting files — Databricks generates Iceberg metadata asynchronously alongside the Delta metadata
CREATE TABLE sp_catalog.ecomm.sales_analytics_uniform (c1 INT)
TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true',
  'delta.enableIcebergCompatV3' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
);

In [0]:
df.write.format("iceberg").mode("append") \
    .saveAsTable("sp_catalog.ecomm_silver_iceberg.sales_analytics_iceberg")

In [0]:
# writing to Iceberg via streaming works; reading from managed Iceberg as a streaming source does not (yet).
# UniForm-Iceberg is the practical route
df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3://sp-db26-eu-west2-test/checkpoints/sales/") \
    .toTable("sp_catalog.ecomm.sales_analytics")